In [13]:
import numpy as np
import sys

sys.path.append('../argschk')
from argschk import argschk

kwargs = {
    'maxiters': 80,
    'algorithm': 'vb',
    'uniquesv': 0,
    'rmsstop': [80, 2.220446049250313e-16, 2.220446049250313e-16],
    'cfstop': [80, 0, 0],
    'minangle': 0,
}

opts = {
    'init': 'random',
    'maxiters': 1000,
    'bias': 1,
    'uniquesv': 0,
    'autosave': 600,
    'filename': 'pca_f_autosave',
    'minangle': 1e-8,
    'algorithm': 'vb',
    'niter_broadprior': 100,
    'earlystop': 0,
    'rmsstop': np.array([100, 1e-4, 1e-3]),  # () means no rms stop criteria
    'cfstop': np.array([]),  # () means no cost stop criteria
    'verbose': 1,
    'xprobe': np.array([]),
    'rotate2pca': 1,
    'display': 0,
}

# Run argschk
opts, wrnmsg = argschk(opts, **kwargs)

# Convert numpy arrays to lists for display
def convert_numpy_to_list(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {key: convert_numpy_to_list(value) for key, value in obj.items()}
    return obj

opts_readable = convert_numpy_to_list(opts)

# Display the options
print("Options after argschk:")
for key, value in opts_readable.items():
    print(f"{key}: {value}")

# Log warnings, if any
if wrnmsg:
    print(f"Warning in Python: {wrnmsg}")


Options after argschk:
init: random
maxiters: 80
bias: 1
uniquesv: 0
autosave: 600
filename: pca_f_autosave
minangle: 0
algorithm: vb
niter_broadprior: 100
earlystop: 0
rmsstop: [80, 2.220446049250313e-16, 2.220446049250313e-16]
cfstop: [80, 0, 0]
verbose: 1
xprobe: []
rotate2pca: 1
display: 0


In [15]:
# return dictionary with A, S, Mu, V, cv, hp, lc
# A: Likely represents a matrix of principal components (loadings).# A is the liner transition matrix - Josh
# S: Another output matrix, possibly the transformed data in the new PCA space (scores). # this is the matrix of prinicipal components
# Mu: The mean values of the data. # this is the bias, like the y intecept in a linear regression, but for a matrix
# V: Variance, possibly of the noise.
# cv: Posterior covariances (likely a structure with details for A and S). # as well as Mu
# hp: Hyperparameters (related to prior distributions).
# lc: Learning curves (metrics for performance over iterations).

# X: The input data matrix.
# ncomp: The number of principal components to compute.
# kwargs: Stands for "keyword arguments". It allows you to pass optional parameters as a dictionary.
import numpy as np
from scipy.io import savemat
import scipy.linalg
import scipy.io
from scipy.sparse import issparse, csr_matrix
import numpy as np
from scipy.io import loadmat
from scipy.linalg import orth  # Use orth from scipy.linalg
import matplotlib.pyplot as plt
import time
from scipy.linalg import subspace_angles
import sys
sys.path.append('../add_m_cols')
from add_m_cols import add_m_cols 

sys.path.append('../add_m_rows')
from add_m_rows import add_m_rows 

sys.path.append('../argschk')
from argschk import argschk

sys.path.append('../cf_full')
from cf_full import cf_full

sys.path.append('../converg_check')
from converg_check import converg_check

sys.path.append('../compute_rms')
from compute_rms import compute_rms 

sys.path.append('../miscomb')
from miscomb import miscomb

sys.path.append('../rmempty')
from rmempty import rmempty

sys.path.append('../subtract_mu')
import subtract_mu_from_sparse


def pca_full(X, ncomp, **kwargs):

    opts = { 'init':'random',
    'maxiters':1000,
    'bias':1,
    'uniquesv':0,
    'autosave':600,
    'filename':'pca_f_autosave',
    'minangle':1e-8,
    'algorithm':'vb',
    'niter_broadprior':100,
    'earlystop':0,
    'rmsstop':np.array([100, 1e-4, 1e-3]), # () means no rms stop criteria
    'cfstop':np.array([]), # () means no cost stop criteria
    'verbose':1,
    'xprobe':np.array([]),
    'rotate2pca':1,
    'display':0 }

    opts, wrnmsg = argschk(opts, **kwargs)
    
    if wrnmsg:
        print(f"warning: {wrnmsg}")

    algorithmVal = opts["algorithm"]
    if (algorithmVal == "ppca"):
        use_prior = False
        use_postvar = False
    elif (algorithmVal == "map"):
        use_prior = True
        use_postvar = False
    elif (algorithmVal == "vb"):
        use_prior = True
        use_postvar = True
    else:
        raise ValueError(f"Wrong value if the argument 'algorithm': {algorithmVal}")

    n1x, n2x = X.shape
    import numpy as np

    # Log and save function parameters
    print("Parameters for rmempty in Python:")
    print(f"X shape: {X.shape}")
    print(f"Xprobe: {opts['xprobe']}")  # If it's an array, log its size
    print(f"opts['init']: {opts['init']}")
    print(f"opts['verbose']: {opts['verbose']}")
    
    # Save X for MATLAB
    np.savetxt("python_X.csv", X, delimiter=",")
    # Save xprobe only if it exists
    if opts['xprobe'].size > 0:
        np.savetxt("python_Xprobe.csv", opts['xprobe'], delimiter=",")
